In [1]:
import pandas as pd

In [2]:
raw = pd.read_csv("../data/raw/complaints.csv", low_memory=False, dtype={"ZIP code": str})
df = raw.copy()
print("Rows at start:", len(df))

Rows at start: 99709


#dtype={"ZIP code": str} keeps ZIP codes as text, so a ZIP starting with 0 doesn’t lose its zero.

In [4]:
#Rename colums to simple names
df.columns = (df.columns
              .str.strip()
              .str.lower()
              .str.replace(" ", "_")
              .str.replace("?", "", regex=False)
              .str.replace("-", "_"))
df.columns.tolist()

['date_received',
 'product',
 'sub_product',
 'issue',
 'sub_issue',
 'company_public_response',
 'company',
 'state',
 'zip_code',
 'tags',
 'submitted_via',
 'date_sent_to_company',
 'company_response_to_consumer',
 'timely_response',
 'complaint_id']

In [5]:
#dates in text to real dates
df["date_received"] = pd.to_datetime(df["date_received"], utc=True).dt.tz_localize(None).dt.normalize()
df["date_sent_to_company"] = pd.to_datetime(df["date_sent_to_company"], utc=True).dt.tz_localize(None).dt.normalize()
df[["date_received", "date_sent_to_company"]].head()

,date_received,date_sent_to_company
0,2023-12-31,2023-12-31
1,2024-01-01,2024-01-01
2,2024-01-01,2024-01-01
3,2024-01-01,2024-01-01
4,2023-12-31,2023-12-31


In [8]:
#Tidy the text columns
text_cols = ["product", "sub_product", "issue", "sub_issue", "company",
             "company_public_response", "tags", "submitted_via",
             "company_response_to_consumer", "timely_response"]
for c in text_cols:
    df[c] = df[c].str.strip()

df["company"] = df["company"].str.upper()

In [7]:
#Handling emplty values
df["sub_issue"] = df["sub_issue"].fillna("Unknown")
df["company_public_response"] = df["company_public_response"].fillna("No public response")
df["tags"] = df["tags"].fillna("None")
df.isna().sum()

date_received                   0
product                         0
sub_product                     0
issue                           0
sub_issue                       0
company_public_response         0
company                         0
state                           0
zip_code                        0
tags                            0
submitted_via                   0
date_sent_to_company            0
company_response_to_consumer    0
timely_response                 0
complaint_id                    0
dtype: int64

In [9]:
#Handling hidden zipcodes
df["zip_hidden"] = df["zip_code"].str.contains("X", na=False)
df["zip3"] = df["zip_code"].str[:3]
df["zip_hidden"].value_counts()

zip_hidden
False    85539
True     14170
Name: count, dtype: int64

Some ZIPs look like 720XX, with the last digits hidden for privacy. zip3 (the first 3 digits) works for every row, so we’ll use it for the ZIP area question.

In [10]:
#New columns
df["year"] = df["date_received"].dt.year
df["month"] = df["date_received"].dt.to_period("M").astype(str)
df["days_to_company"] = (df["date_sent_to_company"] - df["date_received"]).dt.days
df["is_credit_reporting"] = df["product"].str.contains("Credit reporting", na=False)
df[["year", "month", "days_to_company", "is_credit_reporting"]].head()

,year,month,days_to_company,is_credit_reporting
0,2023,2023-12,0,False
1,2024,2024-01,0,True
2,2024,2024-01,0,True
3,2024,2024-01,0,False
4,2023,2023-12,0,False


In [11]:
print("Rows now:", len(df), "| Rows removed:", len(raw) - len(df))
print("Duplicate IDs:", df["complaint_id"].duplicated().sum())
print("Date range:", df["date_received"].min().date(), "to", df["date_received"].max().date())
print("Missing values left:", df.isna().sum().sum())
print(df["days_to_company"].describe())

Rows now: 99709 | Rows removed: 0
Duplicate IDs: 0
Date range: 2023-12-31 to 2026-11-04
Missing values left: 0
count    99709.000000
mean         0.308117
std          3.474881
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        196.000000
Name: days_to_company, dtype: float64


In [12]:
import os
os.makedirs("../data/processed", exist_ok=True)
df.to_parquet("../data/processed/complaints_clean.parquet", index=False)